# 19A4D — Final Frozen Cycle-25 AIA Evaluation (2021–2025)

## Purpose

Combine the five already-generated frozen Cycle-25 prediction shards and compute the first complete independent Cycle-25 scorecard.

**No tuning is permitted in this notebook.**

The following were frozen before Cycle-25 evaluation:
- CNN-GRU weights;
- preprocessing / normalisation;
- Platt calibration parameters;
- operating threshold.

This notebook first verifies all five prediction shards and protocol records. Metrics are computed **only after** the full 49,329-row test set passes all integrity checks.

## Frozen test population

| Year | Targets | Positives |
|---|---:|---:|
| 2021 | 5,104 | 126 |
| 2022 | 9,999 | 119 |
| 2023 | 12,392 | 278 |
| 2024 | 12,553 | 1,316 |
| 2025 | 9,281 | 512 |
| **Total** | **49,329** | **2,351** |

## Metrics

- ROC-AUC
- PR-AUC
- Brier score
- log loss
- TSS
- HSS
- precision
- recall
- F1
- confusion matrix
- prevalence and PR-lift over prevalence

Year-wise metrics are descriptive only; they are not used for model selection or tuning.


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

HOME = Path.home()
PRED_ROOT = HOME / "aia19_cycle25_predictions"
OUT = HOME / "aia19_cycle25_final_evaluation"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    2021: {"targets": 5104, "positives": 126},
    2022: {"targets": 9999, "positives": 119},
    2023: {"targets": 12392, "positives": 278},
    2024: {"targets": 12553, "positives": 1316},
    2025: {"targets": 9281, "positives": 512},
}

EXPECTED_TOTAL = 49329
EXPECTED_POSITIVES = 2351
EXPECTED_THRESHOLD = 0.030438695842933242
CSV_FLOAT_TOL = 1e-15  # permits decimal CSV parse round-off only
EXPECTED_MODEL_SHA256 = "11dc35e089101c8b79d2a6ba6f82d02cdb470d583552cad73e6071016c2a2d76"
EXPECTED_PLATT_COEF = 0.5846760189574352
EXPECTED_PLATT_INTERCEPT = -3.6728404851652265

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


## 1. Verify all five frozen prediction shards and protocol records

### CSV floating-point verification note

The frozen threshold is written to compressed CSV prediction shards and read back as IEEE-754 floating-point values. A decimal text round-trip may change the final binary representation by a few units in the last place even though the numeric threshold is unchanged scientifically.

Therefore the shard check:
- requires the CSV-read threshold to be within `1e-15` of the frozen value;
- separately verifies each protocol record against the frozen threshold;
- recomputes every saved binary prediction using the canonical frozen threshold and requires exact agreement.

This permits serialization round-off only; it does **not** permit threshold reselection or tuning.


In [ ]:
frames = []
protocols = []
inventory = []

required_cols = {
    "target_sample_id",
    "stored_year",
    "region_component_id",
    "HARPNUM",
    "y_true",
    "raw_logit",
    "raw_probability",
    "calibrated_probability",
    "frozen_threshold",
    "frozen_prediction",
}

for year, exp in EXPECTED.items():
    pred_path = PRED_ROOT / f"cycle25_{year}_frozen_predictions.csv.gz"
    proto_path = PRED_ROOT / f"cycle25_{year}_protocol_record.json"

    if not pred_path.exists():
        raise FileNotFoundError(f"Missing prediction shard: {pred_path}")
    if not proto_path.exists():
        raise FileNotFoundError(f"Missing protocol record: {proto_path}")

    df = pd.read_csv(pred_path)
    proto = json.loads(proto_path.read_text())

    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise RuntimeError(f"{year}: missing columns {sorted(missing_cols)}")

    assert len(df) == exp["targets"], (year, len(df), exp["targets"])
    assert df["target_sample_id"].nunique() == exp["targets"], year
    assert int(df["y_true"].sum()) == exp["positives"], year
    assert set(df["stored_year"].unique()) == {year}, year
    assert int(df.isna().sum().sum()) == 0, year

    numeric = df[
        ["raw_logit","raw_probability","calibrated_probability","frozen_threshold","frozen_prediction"]
    ].to_numpy()
    assert np.isfinite(numeric).all(), year

    threshold_vals = df["frozen_threshold"].to_numpy(float)
    threshold_max_abs_diff = float(np.max(np.abs(threshold_vals - EXPECTED_THRESHOLD)))
    print(f"{year}: threshold max |CSV-read - frozen| = {threshold_max_abs_diff:.3e}")
    assert threshold_max_abs_diff <= CSV_FLOAT_TOL, (year, threshold_max_abs_diff)
    assert set(df["frozen_prediction"].unique()).issubset({0,1}), year

    # Verify the saved predictions still match the frozen threshold exactly.
    recomputed_pred = (df["calibrated_probability"].to_numpy(float) >= EXPECTED_THRESHOLD).astype(int)
    assert np.array_equal(recomputed_pred, df["frozen_prediction"].to_numpy(int)), year

    # Verify protocol lineage / no post-test tuning.
    assert proto["year"] == year
    assert proto["targets"] == exp["targets"]
    assert proto["positives"] == exp["positives"]
    assert proto["base_model_sha256"] == EXPECTED_MODEL_SHA256
    assert abs(float(proto["platt_coefficient"]) - EXPECTED_PLATT_COEF) < 1e-12
    assert abs(float(proto["platt_intercept"]) - EXPECTED_PLATT_INTERCEPT) < 1e-12
    assert abs(float(proto["frozen_threshold"]) - EXPECTED_THRESHOLD) < 1e-15
    assert proto["model_weights_updated"] is False
    assert proto["calibrator_refit"] is False
    assert proto["threshold_reselected"] is False
    assert proto["performance_metrics_computed"] is False
    assert proto["cycle25_used_for_tuning"] is False

    actual_sha = sha256_file(pred_path)
    assert actual_sha == proto["prediction_sha256"], (year, actual_sha, proto["prediction_sha256"])

    inventory.append({
        "year": year,
        "rows": len(df),
        "positives": int(df["y_true"].sum()),
        "prediction_sha256": actual_sha,
        "protocol_status": proto["status"],
    })

    frames.append(df)
    protocols.append(proto)

inventory_df = pd.DataFrame(inventory)
print(inventory_df.to_string(index=False))

print("\nALL_YEAR_SHARDS_PASS")


## 2. Concatenate and verify the complete frozen Cycle-25 test population

In [ ]:
all_df = pd.concat(frames, ignore_index=True)

print("Rows:", len(all_df))
print("Unique target IDs:", all_df["target_sample_id"].nunique())
print("Positives:", int(all_df["y_true"].sum()))
print("Years:", sorted(all_df["stored_year"].unique().tolist()))
print("Missing values:", int(all_df.isna().sum().sum()))

assert len(all_df) == EXPECTED_TOTAL
assert all_df["target_sample_id"].nunique() == EXPECTED_TOTAL
assert int(all_df["y_true"].sum()) == EXPECTED_POSITIVES
assert sorted(all_df["stored_year"].unique().tolist()) == [2021,2022,2023,2024,2025]
assert int(all_df.isna().sum().sum()) == 0

# Ensure target IDs do not overlap across year shards.
dup = all_df["target_sample_id"].duplicated(keep=False)
assert not dup.any(), all_df.loc[dup, ["target_sample_id","stored_year"]].head()

combined_path = OUT / "cycle25_2021_2025_frozen_predictions.csv.gz"
all_df.to_csv(combined_path, index=False, compression="gzip")
combined_sha = sha256_file(combined_path)

print("\nCYCLE25_FROZEN_PREDICTION_SET_PASS")
print("Combined prediction SHA256:", combined_sha)


## 3. Metric helpers

TSS is computed at the **already-frozen Cycle-24 threshold** carried in every prediction row.

No test-set threshold search occurs here.


In [ ]:
def hss2(tn, fp, fn, tp):
    num = 2 * (tp*tn - fn*fp)
    den = ((tp+fn)*(fn+tn) + (tp+fp)*(fp+tn))
    return float(num/den) if den else float("nan")

def score(df):
    y = df["y_true"].to_numpy(int)
    p = df["calibrated_probability"].to_numpy(float)
    yhat = df["frozen_prediction"].to_numpy(int)

    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()

    prevalence = float(y.mean())
    pr_auc = float(average_precision_score(y, p))

    return {
        "n": int(len(df)),
        "positives": int(y.sum()),
        "prevalence": prevalence,
        "roc_auc": float(roc_auc_score(y, p)),
        "pr_auc": pr_auc,
        "pr_lift_over_prevalence": float(pr_auc / prevalence) if prevalence > 0 else float("nan"),
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, np.clip(p,1e-7,1-1e-7), labels=[0,1])),
        "threshold": EXPECTED_THRESHOLD,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "tss": float(tp/(tp+fn) - fp/(fp+tn)),
        "hss": hss2(tn, fp, fn, tp),
        "precision": float(precision_score(y, yhat, zero_division=0)),
        "recall": float(recall_score(y, yhat, zero_division=0)),
        "f1": float(f1_score(y, yhat, zero_division=0)),
    }


## 4. Overall independent Cycle-25 scorecard

In [ ]:
overall = score(all_df)

print(json.dumps(overall, indent=2))

(OUT / "cycle25_overall_metrics.json").write_text(
    json.dumps(overall, indent=2) + "\n"
)


## 5. Descriptive year-wise scorecard

These year-wise metrics are reported only to characterize temporal variation within the already-frozen test set.
They are **not** used to modify the model, calibrator, preprocessing, or threshold.


In [ ]:
year_rows = []

for year in sorted(EXPECTED):
    m = score(all_df[all_df["stored_year"].eq(year)])
    year_rows.append({"year": year, **m})

year_metrics = pd.DataFrame(year_rows)
year_metrics.to_csv(OUT / "cycle25_yearwise_metrics.csv", index=False)

cols = [
    "year","n","positives","prevalence","roc_auc","pr_auc",
    "brier","log_loss","tss","hss","precision","recall","f1",
    "tn","fp","fn","tp"
]
print(year_metrics[cols].to_string(index=False))


## 6. Save final frozen-evaluation protocol

In [ ]:
protocol = {
    "status": "FROZEN_TEMPORAL_AIA_PIPELINE_EVALUATED_ON_INDEPENDENT_CYCLE25_2021_2025",
    "test_years": [2021,2022,2023,2024,2025],
    "n": int(len(all_df)),
    "positives": int(all_df["y_true"].sum()),
    "base_model_sha256": EXPECTED_MODEL_SHA256,
    "platt_coefficient": EXPECTED_PLATT_COEF,
    "platt_intercept": EXPECTED_PLATT_INTERCEPT,
    "frozen_threshold": EXPECTED_THRESHOLD,
    "combined_prediction_file": str(combined_path),
    "combined_prediction_sha256": combined_sha,
    "overall_metrics": overall,
    "yearwise_metrics_file": str(OUT / "cycle25_yearwise_metrics.csv"),
    "model_weights_updated_after_cycle25": False,
    "calibrator_refit_after_cycle25": False,
    "threshold_reselected_after_cycle25": False,
    "cycle25_used_for_tuning": False,
    "post_test_tuning_permitted": False,
    "scientific_clearance": False,
    "notes": [
        "All five yearly prediction shards were generated with the same frozen Cycle-24 model, normalisation, Platt calibrator and operating threshold.",
        "Metrics were computed only after the complete 49,329-row prediction set passed integrity checks.",
        "Year-wise metrics are descriptive and were not used for model selection or tuning.",
        "Earlier historical-source and label-clearance limitations remain inherited."
    ],
}

(OUT / "protocol_record.json").write_text(json.dumps(protocol, indent=2) + "\n")
inventory_df.to_csv(OUT / "cycle25_prediction_shard_inventory.csv", index=False)

print(json.dumps(protocol, indent=2))
print("\n19A4_FINAL_EVALUATION_COMPLETE")
